# Within-subject mixed-effects factorial models

Tests whether participants' gait responds *additively* to three task prompts or
whether the prompts interact i.e., whether the cost of one objective
depends on what the others are asking for.

**Design.** Every participant walked under all 27 combinations of three prompts,
each at three levels:

* `Speed` — slow / typical / fast (ordinal, 0–2)
* `Accuracy` — ignore / near / accurate (levels are not equally spaced, so this
  enters the models as a categorical factor)
* `Balance` — zero / low / high perturbation (ordinal, 0–2)

**Outcomes**, each fit in its own model: walking speed, foot-placement error,
step width and length and their variability, head angle, and metabolic cost.

**Model hierarchy**, all with a subject-specific random intercept `(1 | Subject)`:

| Model | Fixed effects | Question |
| --- | --- | --- |
| M1 | `Speed + Accuracy + Balance` | purely additive control |
| M2 | M1 + all pairwise interactions | do objectives interact? |
| M3 | `Speed * Accuracy * Balance` | is there three-way structure? |

M1 vs. M2 tests evidence against additive control; M2 vs. M3 tests for
higher-order structure. All fits use maximum likelihood (not REML) so the
likelihood-ratio tests are valid across differing fixed-effects structures.

**Input.** `data/data_BMH*.xlsx`, one workbook per participant.
Run the notebook top to bottom from the repository root.

## Setup

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

# statsmodels emits convergence warnings for the saturated models; the fits are
# checked via the random-effect variance and scale printed alongside each result.
warnings.filterwarnings("ignore")

# Paths are relative to the repository root, so launch Jupyter from there.
DATA_DIR = Path("data")
GAIT_METRICS_DIR = DATA_DIR

## Load and prepare the data

One workbook per participant, concatenated into a single long dataframe. Column
names are made formula-safe, prompt levels are mapped to 0/1/2, and the distance
metrics are converted from millimetres to metres.

In [ ]:
# Subject workbooks. Energy expenditure is read from the mass-normalized
# "EE Wkg" column below.
bmh_files = [
    'data_BMH01.xlsx', 'data_BMH02.xlsx', 'data_BMH06.xlsx', 'data_BMH07.xlsx', 'data_BMH08.xlsx','data_BMH09.xlsx',
    'data_BMH10.xlsx', 'data_BMH13.xlsx','data_BMH17.xlsx', 'data_BMH19.xlsx', 'data_BMH20.xlsx','data_BMH21.xlsx'
]

performance_metrics_raw = ['Mean Error Straights', 'Mean Width Straights (mm)', 'Straights Width Variability (mm)',
    'Mean Length Straights (mm)', 'Straights Length Variability (mm)', 'Average Speed (m/s)', 'Head Angle (deg)']

performance_metrics = [
    col.replace(" ", "_")
       .replace("(", "")
       .replace(")", "")
       .replace("/", "_per_")
    for col in performance_metrics_raw
]
rename_map_metrics = dict(zip(performance_metrics_raw, performance_metrics))

# Prompt mappings
speed_mapping = {'Slow': 0, 'Medium': 1, 'Fast': 2}
accuracy_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
balance_mapping = {'Low': 0, 'Medium': 1, 'High': 2}

# Generate data frame for all subjects
all_data = []
for bmh_file in bmh_files:
    # Pull subject file
    data_path = GAIT_METRICS_DIR / bmh_file
    df_data = pd.read_excel(data_path)

    # Map condition prompt as numeric levels [0, 1, 2]
    df_data['Speed'] = df_data['Walking Speed'].map(speed_mapping).astype(int)
    df_data['Accuracy'] = df_data['Accuracy'].map(accuracy_mapping).astype(int)
    df_data['Balance'] = df_data['Balance'].map(balance_mapping).astype(int)
    df_data['Condition'] = df_data.apply(lambda row: f"s{row['Walking Speed']}a{row['Accuracy']}b{row['Balance']}", axis=1)

    # Add task performance metrics
    df_model = df_data[performance_metrics_raw].copy()
    df_model.rename(columns=rename_map_metrics, inplace=True)

    # Add condition prompts
    df_model['Speed'] = df_data['Speed'].values
    df_model['Accuracy'] = df_data['Accuracy'].values
    df_model['Balance'] = df_data['Balance'].values

    # Add energy expenditure as fourth metric (now pulled from W/kg column)
    df_model["EE"] = df_data['EE Wkg'].values

    # Add subject ID
    subject_id = bmh_file.replace("data_", "").replace(".xlsx", "")
    df_model["Subject"] = subject_id

    # Append data frame of this subject
    all_data.append(df_model)

# Merge all subject data frames
df_all = pd.concat(all_data, ignore_index=True)

# Convert error/variability metrics from mm to m (source Excel columns are in mm;
# column names below still carry the original "_mm" suffix, but values are now in meters).
# Unit is a linear rescale, so R2m/R2c and LRT stat/df/p are unaffected -- this only
# changes the scale of AIC/BIC/logLik and any raw coefficients.
mm_to_m_cols = ["Mean_Error_Straights", "Straights_Width_Variability_mm", "Straights_Length_Variability_mm"]
df_all[mm_to_m_cols] = df_all[mm_to_m_cols] / 1000.0

# Log transform of step width variability, for outcomes where the raw metric is
# too skewed for a Gaussian model.
df_all["log_SWV"] = np.log(df_all["Straights_Width_Variability_mm"])

# Collapse Accuracy to 2 levels: Ignore (0) vs. Near+Accurate merged (1)
# Near and Accurate were found to be behaviorally similar (not equally spaced
# from Ignore), so treating all 3 levels as an unordered categorical factor
# wastes a df on a near-null Near-vs-Accurate contrast and dilutes power.
df_all["Accuracy_2lvl"] = df_all["Accuracy"].map({0: 0, 1: 1, 2: 1}).astype(int)

## Model-fitting helpers

`run_staged_models()` is the entry point: it fits M1–M3 for each outcome and
returns both a fit table (marginal and conditional R², AIC, BIC) and the
likelihood-ratio tests between successive models.

In [ ]:
# Optimizers tried in order. "lbfgs" is what the reported fits used, but on
# recent scipy/statsmodels releases it stalls on the boundary of the
# random-effect variance for several outcomes: the fit reports convergence with
# cov_re = 0 and an infinite log-likelihood (statsmodels >= 0.15 instead raises
# LinAlgError while inverting the Hessian). That is what silently turned the
# likelihood-ratio table into inf/NaN. The other optimizers find the same
# optimum, so falling back to them reproduces the reported numbers.
MIXEDLM_METHODS = ("lbfgs", "bfgs", "powell")
_FALLBACK_REPORTED = []


def fit_mixedlm_ml(formula: str, data: pd.DataFrame, group_col: str,
                   re_formula: str = "1", methods=MIXEDLM_METHODS):
    """
    Fit a MixedLM using maximum likelihood (ML) for model comparison.
    re_formula='1' gives a random intercept for each group.

    Tries each optimizer in `methods` and returns the first fit with a finite
    log-likelihood, so a boundary fit from one optimizer cannot silently
    poison the AIC/BIC/LRT columns downstream.
    """
    last_problem = None

    for method in methods:
        model = smf.mixedlm(formula, data=data, groups=data[group_col], re_formula=re_formula)

        # Max likelihood (ML) fit for likelihood ratio tests
        # Cannot use restricted max likelihood (REML) across models with different fixed effects structures
        try:
            result = model.fit(reml=False, method=method, maxiter=2000, full_output=True, disp=False)
            llf = result.llf
        except Exception as err:      # e.g. singular Hessian on statsmodels >= 0.15
            last_problem = f"{method}: {type(err).__name__}: {err}"
            continue

        if not np.isfinite(llf):
            last_problem = f"{method}: non-finite log-likelihood ({llf})"
            continue

        if method != methods[0] and not _FALLBACK_REPORTED:
            print(
                f"[note] optimizer '{methods[0]}' did not give a usable fit "
                f"({last_problem}); refitting with '{method}'. See the README."
            )
            _FALLBACK_REPORTED.append(method)

        # AIC = (-2 * log-liklihood) + (2 * no. params)
        # Penalizes complexity, rewards fit s.t. lower AIC = better out-of-sample prediction
        # BIC = (-2 * log-liklihood) + (no. params) * log (no. observations)
        # Penalizes complexity more strongly s.t. lower BIC = better under Bayes approximation
        return result

    raise RuntimeError(
        f"No optimizer in {methods} produced a usable fit for: {formula} "
        f"(last problem -- {last_problem})"
    )
    
def likelihood_ratio_test(res_small, res_big):
    """
    Compare goodness of fit between two statistical models
    "big" MixedLM (more params) vs. "small" MixedLM (fewer params)
    Assumes both fits used ML (reml=False) and same random-effects structure.
    """
    # Get log-likelihood for each model
    llf_small, llf_big = res_small.llf, res_big.llf

    # Get number of estimated params for each model
    df_small, df_big = res_small.df_modelwc, res_big.df_modelwc

    # Compute likelihood ratio statistic and p-value
    lr_stat = 2 * (llf_big - llf_small)     # difference between log-likelihoods
    df_diff = int(df_big - df_small)        # difference between number of param

    p_value = stats.chi2.sf(lr_stat, df_diff) if df_diff > 0 else np.nan
    return lr_stat, df_diff, p_value

def fixed_fit_and_resid(res, df, y_col):
    """
    Compute population-level fitted values and residuals.
    Uses fixed effects only (res.predict(df)).
    """
    y_hat = np.asarray(res.predict(df)).reshape(-1)
    y_obs = df[y_col].to_numpy()
    resid = y_obs - y_hat
    return y_obs, y_hat, resid

def rsq_nakagawa(mixedlm_result, df, y_col):
    """
    Compute the marginal and conditional r-squared values
    """

    # Fixed-effects prediction (population-level)
    y_hat_fixed = mixedlm_result.predict(df)

    # Variance components
    var_fixed = np.var(y_hat_fixed, ddof=1)
    var_resid = mixedlm_result.scale

    # Random-effects variance (intercept-only)
    re_var = mixedlm_result.cov_re.iloc[0, 0]

    # Maginal R² = variance explained by fixed effects / total variance
    rsq_marginal = var_fixed / (var_fixed + re_var + var_resid)

    # Conditional R² = variance explained by fixed + random effects / total variance
    rsq_conditional = (var_fixed + re_var) / (var_fixed + re_var + var_resid)

    return rsq_marginal, rsq_conditional

def run_staged_models(data, outcomes, group_col="Subject", re_formula="1"):
    """
    Compare nested mixed-effects models using likelihood ratio tests and save statistics
    """
    rows = []
    lrt_rows = []

    for y in outcomes:
        # Set up three models with different fixed effects structures
        m1 = f"{y} ~ Speed + C(Accuracy) + Balance"
        m2 = f"{y} ~ Speed + C(Accuracy) + Balance + Speed:C(Accuracy) + Speed:Balance + C(Accuracy):Balance" 
        m3 = f"{y} ~ Speed * C(Accuracy) * Balance"

        # Saturated models can hit a singular random-effects covariance; the
        # fits are still usable for the likelihood-ratio tests, and the
        # resulting statsmodels warnings are silenced in the setup cell.
        r1 = fit_mixedlm_ml(m1, data, group_col, re_formula=re_formula)
        r2 = fit_mixedlm_ml(m2, data, group_col, re_formula=re_formula)
        r3 = fit_mixedlm_ml(m3, data, group_col, re_formula=re_formula)

        # Get marginal and conditional R^2 per model
        rsq1_m, rsq1_c = rsq_nakagawa(r1, data, y)
        rsq2_m, rsq2_c = rsq_nakagawa(r2, data, y)
        rsq3_m, rsq3_c = rsq_nakagawa(r3, data, y)

        # Likelihood ratio tests for each nested model comparison
        lr12, df12, p12 = likelihood_ratio_test(r1, r2)
        lr23, df23, p23 = likelihood_ratio_test(r2, r3)

        # Save stats results
        rows.extend([
            {"Outcome": y, "Model": "M1", "R2_m": rsq1_m, "R2_c": rsq1_c, "AIC": r1.aic, "BIC": r1.bic, "logLik": r1.llf, "n_params": r1.df_modelwc},
            {"Outcome": y, "Model": "M2", "R2_m": rsq2_m, "R2_c": rsq2_c, "AIC": r2.aic, "BIC": r2.bic, "logLik": r2.llf, "n_params": r2.df_modelwc},
            {"Outcome": y, "Model": "M3", "R2_m": rsq3_m, "R2_c": rsq3_c, "AIC": r3.aic, "BIC": r3.bic, "logLik": r3.llf, "n_params": r3.df_modelwc}
        ])

        lrt_rows.extend([
            {"Outcome": y, "Comparison": "M1 vs M2", "LR stat": lr12, "p": p12},
            {"Outcome": y, "Comparison": "M2 vs M3", "LR stat": lr23, "p": p23},
        ])

    return pd.DataFrame(rows), pd.DataFrame(lrt_rows)

## Model comparison

Fits M1–M3 for every outcome and reports the likelihood-ratio tests. A
significant M1 vs. M2 means the additive model is insufficient.

In [ ]:
OUTCOMES = ["EE", "Average_Speed_m_per_s", "Mean_Error_Straights", "Straights_Width_Variability_mm", "log_SWV", 
            "Mean_Width_Straights_mm", "Mean_Length_Straights_mm", "Straights_Length_Variability_mm", "Head_Angle_deg"]


model_fits_table, lrt_results_table = run_staged_models(df_all, OUTCOMES, group_col="Subject", re_formula="1")

print("\nLikelihood ratio test results:")
print(lrt_results_table.to_string(index=False, float_format="%.1e"))

print("\nAll model fits:")
print(model_fits_table)

## Coefficients of the preferred models

For each primary outcome, the model favoured by the tests above is refit and its
terms are tested with a Wald test, which handles the categorical accuracy factor
as a single multi-df term rather than as separate contrasts.

In [ ]:
# Formula chosen per outcome by the likelihood-ratio tests above.
models = [
    "Average_Speed_m_per_s ~ Speed + C(Accuracy) + Balance + Speed:C(Accuracy) + Speed:Balance + C(Accuracy):Balance",
    "Mean_Error_Straights ~ Speed + C(Accuracy) + Balance",
    "Straights_Width_Variability_mm ~ Speed + C(Accuracy) + Balance + Speed:C(Accuracy) + Speed:Balance + C(Accuracy):Balance",
    "EE ~ Speed + C(Accuracy) + Balance",
]

alpha = 0.05  # significance level

for formula in models:
    print("\n==============================")
    print(f"Model: {formula}")

    # Fit model
    rOUT = fit_mixedlm_ml(formula, df_all, group_col="Subject", re_formula="1")

    # Wald test over model terms (keeps categorical predictors as single terms)
    wt_df = rOUT.wald_test_terms(skip_single=False, scalar=False).table

    # statsmodels has used a few different names for the p-value column
    p_col = next((c for c in ["P>chi2", "pvalue", "P>|z|"] if c in wt_df.columns), None)
    if p_col is None:
        print("\n[Warning] Could not find a p-value column in the Wald table.")
        continue

    wt_df = wt_df.copy()
    wt_df["significant"] = wt_df[p_col] < alpha
    print(wt_df.to_string())

    n_sig = int(wt_df["significant"].sum())
    print(f"\n{n_sig} term(s) significant at alpha = {alpha}")